<a href="https://colab.research.google.com/github/yuri-maradini/TempSal/blob/main/src/train_ueyes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning TempSAL su UEyes — Colab

Notebook pronto per lanciare il training vero (Step 4) su GPU, invece che sulla CPU locale.

**Prima di eseguire questo notebook**, su Google Drive crea una cartella (default atteso: `MyDrive/TempSAL_UEyes/`) contenente:
- `multilevel_tempsal.pt` — il checkpoint pre-addestrato originale
- `data_ueyes.zip` — l'archivio di `data_ueyes/` generato in locale (Step 1-2)

Poi: **Runtime → Cambia tipo di runtime → GPU**, prima di eseguire le celle.

In [1]:
# Controllo che sia stata assegnata una GPU
!nvidia-smi

Sat Sep 12 12:12:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Cambia questo path se hai usato un nome/percorso diverso su Drive
DRIVE_DIR = '/content/drive/MyDrive/TempSAL_UEyes'

assert os.path.isdir(DRIVE_DIR), (
    f"Cartella non trovata: {DRIVE_DIR}\n"
    "Creala su Drive e caricaci multilevel_tempsal.pt + data_ueyes.zip prima di continuare."
)
print('Contenuto trovato su Drive:', os.listdir(DRIVE_DIR))

Contenuto trovato su Drive: ['multilevel_tempsal.pt', 'data_ueyes.zip', 'multilevel_tempsal_ueyes.pt', 'multilevel_tempsal_ueyes_v2.pt', 'multilevel_tempsal_ueyes_v3.pt', 'multilevel_tempsal_ueyes_v4.pt']


In [4]:
# Codice: sempre aggiornato da GitHub, non serve preparazione
!git clone https://github.com/yuri-maradini/TempSal.git /content/TempSal

Cloning into '/content/TempSal'...
remote: Enumerating objects: 264, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 264 (delta 104), reused 215 (delta 58), pack-reused 0 (from 0)
Receiving objects: 100% (264/264), 10.32 MiB | 25.11 MiB/s, done.
Resolving deltas: 100% (104/104), done.


In [5]:
!cd /content/TempSal && git pull


Already up to date.


In [6]:
import shutil

os.makedirs('/content/TempSal/src/checkpoints', exist_ok=True)
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal.pt',
)
# multilevel_tempsal_ueyes_v4.pt e' il checkpoint della quarta run (backbone
# sbloccato, statistiche BatchNorm congelate, il migliore sul ramo temporale
# finora): questa run riparte da li'.
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v4.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v4.pt',
)
print('Checkpoint copiati (originale + v4, il warm-start di questa run).')

Checkpoint copiati (originale + v4, il warm-start di questa run).


In [7]:
import time
import zipfile

# Estratto sul disco locale di Colab (veloce), non lasciato sul mount di Drive
# (l'I/O su Drive montato e' molto piu' lento per tanti file piccoli, e qui
# ce ne sono migliaia tra immagini, mappe e volumi temporali).
t0 = time.time()
with zipfile.ZipFile(f'{DRIVE_DIR}/data_ueyes.zip') as zf:
    zf.extractall('/content/TempSal/')
print(f'Dati estratti in {time.time() - t0:.0f}s')

Dati estratti in 25s


In [8]:
# Controllo veloce di integrita': i conteggi devono combaciare con quelli
# verificati in locale (1872 train / 108 val per ciascuna sottocartella)
for sub in ['images', 'maps', 'fixation_maps', 'saliency_volumes_5', 'fixation_volumes_5']:
    for split in ['train', 'val']:
        d = f'/content/TempSal/data_ueyes/{sub}/{split}'
        n = len(os.listdir(d)) if os.path.isdir(d) else 'MANCANTE'
        print(f'{sub:22s} {split:5s} -> {n}')

images                 train -> 1872
images                 val   -> 108
maps                   train -> 1872
maps                   val   -> 108
fixation_maps          train -> 1872
fixation_maps          val   -> 108
saliency_volumes_5     train -> 9360
saliency_volumes_5     val   -> 540
fixation_volumes_5     train -> 9360
fixation_volumes_5     val   -> 540


In [9]:
# Colab ha gia' PyTorch con supporto CUDA preinstallato: installiamo solo le
# altre dipendenze del progetto, senza toccare torch/torchvision/torchaudio
# (forzare i pin usati in locale, pensati per una build CPU, rischierebbe di
# rimpiazzare la build CUDA gia' pronta di Colab con una incompatibile).
!pip install -q wandb pycocotools ftfy einops clip-anytorch kornia regex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.5 MB/s eta 0:00:00


In [10]:
# Verifica che l'installazione sopra non abbia rovinato il supporto CUDA di torch
import torch
print('torch', torch.__version__, '| CUDA disponibile:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU non disponibile: controlla Runtime > Cambia tipo di runtime > GPU'

torch 2.11.0+cu128 | CUDA disponibile: True


## wandb (consigliato per questa run)

Il primo run (10 epoche) è stato fatto con `WANDB_MODE=disabled`: nessuna curva salvata, i numeri per epoca sono stati recuperati a mano dall'output della cella di training. Per questa run vale la pena accendere il logging vero, così le curve di CC/KLDIV/NSS/SIM (aggregate e per-slice) restano disponibili per il confronto e per la tesi senza dover rileggere l'output della cella.

Esegui la cella sotto (chiede l'API key, la trovi su wandb.ai/authorize) prima di lanciare il training. Se preferisci comunque saltarlo, aggiungi di nuovo `WANDB_MODE=disabled` (o `=offline` per salvare i log in locale senza account) davanti al comando `python train.py` nella cella di training.</cell id="HNcrR_LSgcXM">


In [11]:
import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yurimaradini (yurimaradini-universit-di-padova) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Training (run 6 — learning rate del mixing "intermedio")

**Risultato della run 5** (vedi cronologia nel repo per il razionale originale): con `--mixing_lr 1e-5` (10 volte `--lr 1e-6`, non 100 come scritto per errore nella nota della run 5) il decoder di mixing non si "ricalibra" affatto — smette di migliorare dopo l'epoca 1 e da lì in poi overfitta silenziosamente (train loss in discesa continua per 20 epoche, ma CC/KLDIV/NSS/SIM di validazione tutti stabili o in leggero peggioramento, mentre solo il ramo temporale, che non usa quel learning rate, continua a migliorare leggermente). Il valore usato in run 5 non era arbitrario — era lo stesso learning rate con cui il mixing decoder si era già allenato con successo nelle run 1 e 2 — ma in quel contesto il resto della rete restava congelato: il bersaglio del decoder era fisso. In v4/v5, il bersaglio (le slice temporali di `pnas_vol`) si sposta ad ogni step perché il backbone è sbloccato: un learning rate 10 volte più alto fa sì che il decoder rincorra un bersaglio in movimento con passi troppo grandi, overfittando sul rumore invece di seguirne la tendenza.

**Questa run (l'ultima pianificata per lo Step 4)** testa un valore intermedio invece di scartare l'idea:
- `--mixing_lr 3e-6`: circa 3 volte `--lr 1e-6` (contro le 10 volte di v5) — la media geometrica tra "nessun effetto osservabile" (v4, senza `--mixing_lr`, di fatto 1x) e "overfitting quasi immediato" (v5, 10x). Un passo abbastanza più grande di quello del backbone da permettere una vera ricalibrazione, ma non così grande da ignorare il bersaglio che si sposta.
- `--no_epochs 10` (dimezzato rispetto a v5): la run 5 ha mostrato che l'effetto (positivo o negativo) del mixing_lr si manifesta già entro le prime 1-2 epoche e poi resta stabile o degrada — non serve un budget di 20 epoche per osservarlo, e un budget più corto riduce il rischio di consumare inutilmente quota Colab.
- Warm-start da `multilevel_tempsal_ueyes_v4.pt` (non da v5): la run 5 si è rivelata funzionalmente equivalente a v4 (nessun miglioramento reale, solo un checkpoint salvato alle epoche 0-1 prima di iniziare a overfittare), quindi si riparte dalla stessa base pulita invece che da una run che non ha aggiunto nulla.
- Tutto il resto invariato: `--train_enc 1`, `--train_model 1`, `--lr 1e-6` per backbone e testa temporale, `--batch_size 16 --grad_accum_steps 2`, `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`.

**Da controllare dopo la run**: stesso criterio di v5 — se la mappa aggregata (CC/KLDIV di validazione) supera il livello di v2/v4 senza che il train/val gap si allarghi, l'ipotesi del "mixing decoder in ritardo" è confermata con un learning rate più moderato; se anche a 3x si osserva lo stesso pattern di v5 (train loss giù, validazione piatta o peggiore), l'ipotesi va scartata definitivamente e lo Step 4 si chiude con v4 come checkpoint finale.

In [ ]:
%cd /content/TempSal/src
# PYTORCH_CUDA_ALLOC_CONF: riduce la frammentazione dell'allocatore, utile
# ora che la memoria e' molto piu' vicina al limite della T4 (vedi cella
# markdown sulla run 4 sull'OOM del primo tentativo).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py \
  --enc_model pnas_boosted_multi \
  --dataset_dir ../data_ueyes/ \
  --model_path ./checkpoints/multilevel_tempsal_ueyes_v4.pt \
  --model_vol_path ./checkpoints/multilevel_tempsal_ueyes_v4.pt \
  --train_model 1 \
  --train_enc 1 \
  --lr 1e-6 \
  --mixing_lr 3e-6 \
  --batch_size 16 \
  --grad_accum_steps 2 \
  --no_epochs 10 \
  --model_val_path ./checkpoints/multilevel_tempsal_ueyes_v6.pt

In [ ]:
# Copia il checkpoint fine-tuned su Drive, cosi' sopravvive alla chiusura
# della sessione Colab. Puoi rieseguire questa cella anche a training ancora
# in corso, per avere un backup intermedio.
import shutil

src_ckpt = '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v6.pt'
dst_ckpt = f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v6.pt'
shutil.copy(src_ckpt, dst_ckpt)
print('Copiato su Drive:', dst_ckpt)